In [ ]:
arquivo = r"C:\TCC\Documentos\Instancias\Instância Teste 5 jobs.txt"

In [ ]:
from pyscipopt import Model, quicksum
from itertools import combinations

In [ ]:
def ler_instancia(nome_arquivo):
    with open(nome_arquivo, "r", encoding="utf-8") as f:
        linhas = [linha.strip() for linha in f if linha.strip()]

    peso = {}
    familia_produto = {}
    capacidade = {}
    tempo_proc = {}
    maquinas_familia = {}
    familia = {}
    produtos_por_familia = {}

    inst = None
    secao = None
    produto_id = 1

    for linha in linhas:

        if "NÚMERO DE PRODUTOS" in linha:
            texto, inst = linha.split(":")

        # identificar seção
        if "NOME DO PRODUTO" in linha:
            secao = "produtos"
            continue

        elif "MÁQUINAS/CAPACIDADE" in linha:
            secao = "maquinas"
            continue

        elif "FAMÍLIA/TEMPO" in linha:
            secao = "familias"
            continue

        elif "---" in linha:
            continue

        # -------------------
        # PRODUTOS
        # -------------------
        if secao == "produtos":
            nome, p, f = linha.split("/")

            peso[produto_id] = float(p)
            familia_produto[produto_id] = int(f)

            produto_id += 1

        # -------------------
        # MÁQUINAS
        # -------------------
        elif secao == "maquinas":
            maq, cap = linha.split("/")

            capacidade[maq] = float(cap)

        # -------------------
        # FAMÍLIAS
        # -------------------
        elif secao == "familias":
            fam, tempo, maq = linha.split("/")

            fam = int(fam)
            familia[fam] = fam
            tempo_proc[fam] = int(tempo)
            maquinas_familia[fam] = maq.split(",")

    for produto, fam in familia_produto.items():

        if fam not in produtos_por_familia:
            produtos_por_familia[fam] = []

        produtos_por_familia[fam].append(produto)

    return (
        inst,
        peso,
        familia_produto,
        capacidade,
        tempo_proc,
        maquinas_familia,
        familia,
        produtos_por_familia
    )

def cria_lotes(capacidade, peso, produtos_por_familia, tempo_proc, familia_produto):

    T = []
    P = []
    lotes_validos = []
    A = []
    maquinas = list(capacidade.keys())
    produtos = list(peso.keys())

    maior_capacidade = max(capacidade.values())

    for fam, produtos_fam in produtos_por_familia.items():
        for r in range(1, len(produtos_fam) + 1):
            for lote in combinations(produtos_fam, r):

                if sum(peso[p] for p in lote) <= maior_capacidade:
                    lotes_validos.append(lote)

    #Binariza :)
    for lotes in lotes_validos:
        linha = [1 if n in lotes else 0 for n in produtos]
        A.append(linha)

    #tempos
    for lote in lotes_validos:
        tempo_lote = tempo_proc[familia_produto[lote[0]]]
        T.append(tempo_lote)

    #pesos
    for lote in lotes_validos:
        peso_lote = sum(peso[p] for p in lote)
        P.append(peso_lote)

    return (
        T,
        P,
        lotes_validos,
        A,
        maquinas,
        produtos
    )

def master(modelo, X, Cmax):
    
    modelo.setObjective(quicksum(X[n] for n in range(len(X))), "minimize")

    modelo.optimize()

    return(X)

def sub(lotes_utilizados, modelo, y, Cmax):

    modelo.setObjective(Cmax, "minimize")

    modelo.optimize()

    return(Cmax)

In [ ]:
def main(nome_arquivo):

    inst, peso, familia_produto, capacidade, tempo_proc, maquinas_familia, familia, produtos_por_familia = ler_instancia(nome_arquivo)

    T, P, lotes_validos, A, maquinas, produtos = cria_lotes(capacidade, peso, produtos_por_familia, tempo_proc, familia_produto)

    lotes_utilizados = []

    # --------------------------------------------------------------------------------------------
    # MASTER
    # --------------------------------------------------------------------------------------------

    master = Model("Master")

    # Variáveis

    X = [None for n in range(len(lotes_validos))]
    for n in range(len(X)):
        X[n] = master.addVar(f"X{n}", "binary")

    # Restrições

    for i in range(len(produtos)):
        con1 = master.addCons(quicksum(X[n] * A[n][i] for n in range(len(lotes_validos))) == 1)

    # --------------------------------------------------------------------------------------------
    # SUB
    # --------------------------------------------------------------------------------------------

    sub = Model("Sub")

    # Variáveis

    y = [[None for j in range(len(maquinas))] for u in range(len(lotes_utilizados))]
    for u in range(len(lotes_utilizados)):
        for j in range(len(maquinas)):
            y[u][j] = sub.addVar(f"Y{u},{j}", "binary")

    Cmax = sub.addVar("Cmax", "continuous")

    # Restrições

    for u, lote in enumerate(lotes_utilizados):
        for j, maquina in enumerate(maquinas):
            if maquina not in maquinas_familia[familia_produto[lote[0]]]:
                con2 = sub.addCons(y[u][j] == 0)

    for j in range(len(maquinas)):
        con3 = sub.addCons(quicksum(y[u][j] * T[u] for u in range(len(lotes_utilizados))) <= Cmax)

    for u in range(len(lotes_utilizados)):
        for j in range(len(maquinas)):
            con4 = sub.addCons(P[u] * y[u][j] <= capacidade[maquinas[j]])

    # -----------------------------------------------------------------------------------------------
    # Resolve
    # -----------------------------------------------------------------------------------------------

    

In [ ]:
if __name__ == '__main__':
    main(arquivo)